# 7장. Omniverse 연결을 위한 AI 추론 서버 준비

## 개요

**목표:** 평가한 모델을 추론 서버로 실행 → 입력 두께로 형상 모핑(morphing) → AI 모델 추론 → 노드별 응력/변위 데이터 수신 → Omniverse 연결 준비

| 순서 | 배울 내용 | 직접 하는 일 |
|---|---|---|
| 1. Omniverse와 추론 서버의 연결 구조 | 서버/클라이언트/API의 관계 | 요청과 응답의 흐름 이해 |
| 2. 모델과 형상 준비하기 | 정규화 기준/두께 모핑/전체 노드와 표면 | 평가한 모델과 모핑 자료 준비 |
| 3. 주고받을 데이터 이해하기 | 두께/하중 입력과 노드별 결과 | JSON 요청 작성 |
| 4. 서버 실행하기 | BAT 파일/주소/포트 | BAT 생성/실행 후 상태 확인 |
| 5. 예측 결과 받기 | 결과 전달과 두께 변화 | 첫 요청 → 전달 확인 → 같은 하중에서 두께 비교 |
| 6. 다음 장 준비하기 | 서버 유지와 연결 정보 | 연결 정보 저장 후 Omniverse 실습 준비 |


## 2. 추론에 사용할 모델과 후크 형상 준비하기


### 2.1 평가한 모델과 정규화 기준 불러오기


### 07장 / 셀 04 / Notebook


In [ ]:
# 1. 교육 폴더와 필요한 함수를 준비합니다.
from pathlib import Path, PureWindowsPath
from datetime import datetime
import json, sys, subprocess
import numpy as np
import torch
working_dir = Path.cwd().resolve()
education_dir = next((p for base in [working_dir, *working_dir.parents]
                      for p in [base, base/'education']
                      if (p/'preparation/serve_transolver.py').is_file()), None)
if education_dir is None:
    raise FileNotFoundError('교육 폴더 또는 그 하위 폴더에서 실행하세요.')
if str(education_dir) not in sys.path:
    sys.path.insert(0, str(education_dir))
from course_utils.inference import load_selected_model, predict_fields
from course_utils.dataset import file_sha256
from course_utils.morphing import prepare_morphing_reference, morph_points, morph_reference_stl
from course_utils.lesson_handoff import resolve_evaluation_handoff

# 6장 마지막에 저장한 연결 정보에서 평가 폴더를 자동으로 찾습니다.
evaluation_dir = resolve_evaluation_handoff(education_dir)
print('6장에서 이어받은 평가 폴더:', evaluation_dir)
# 2. 평가 기록에 연결된 학습 폴더를 찾습니다.
evaluation_report = json.loads((evaluation_dir/'evaluation_report.json').read_text(encoding='utf-8'))
assert evaluation_report['status'] == 'completed' and not evaluation_report['test_used_to_update_model']
expected_context = evaluation_report['context']
training_dir = education_dir/'runs/training'/PureWindowsPath(expected_context['checkpoint']).parent.name
if not torch.cuda.is_available():
    raise RuntimeError('1장의 GPU/CUDA용 PyTorch/커널을 확인하세요.')
# 3. 같은 모델과 정규화 기준을 GPU에 불러옵니다.
device = 'cuda'
model, normalizer, cases, split, fixed, context = load_selected_model(education_dir, training_dir, device)
assert all(context[k] == v for k, v in expected_context.items() if k != 'checkpoint')
assert getattr(model, 'requires_full_point_context', False)
# 4. 이번 서버 준비 자료를 저장할 폴더를 만듭니다.
server_dir = education_dir/'runs/inference_server'/('student_transolver_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
server_dir.mkdir(parents=True, exist_ok=False)
print('추론 모델: PhysicsNeMo Transolver | 선택 에폭:', context['selected_epoch'])
print('6장과 같은 모델 SHA256:', context['checkpoint_sha256'])
print('GPU:', torch.cuda.get_device_name(), '| 장치:', device)
print('이번 서버 준비 폴더:', server_dir.resolve())


### 2.4 모핑 자료 파일 만들기


### 07장 / 셀 07 / Notebook


In [ ]:
surface_dir = server_dir/'surfaces'
morphing_manifest = prepare_morphing_reference(education_dir, training_dir, surface_dir)
assert morphing_manifest['context']['checkpoint_sha256'] == context['checkpoint_sha256']
print('기준 케이스:', morphing_manifest['reference_case_id'])
print('기준 두께(mm):', morphing_manifest['reference_thickness_mm'])
print('연속 두께 범위(mm):', morphing_manifest['thickness_range_mm'])
print('전체 노드:', morphing_manifest['full_node_count'], '| 표면 노드:', morphing_manifest['surface_node_count'])
print('기준 STL:', surface_dir/'reference_hook.stl')
preview_stl = morph_reference_stl(surface_dir, 9.0, surface_dir/'example_9mm.stl')
print('9 mm 모핑 STL 예시:', preview_stl)
print('모핑 자료 폴더:', surface_dir.resolve())


## 3. 서버와 주고받을 데이터 이해하기


### 3.1 두께/하중을 요청으로 표현하기


### 07장 / 셀 10 / Notebook


In [ ]:
request_body = {'thickness_mm': 10.25, 'force_N': 343.561}
print('요청 입력:', json.dumps(request_body, ensure_ascii=False))


## 4. 추론 서버 실행하고 연결 확인하기


### 4.1 서버 실행용 BAT 파일 만들기


### 07장 / 셀 14 / Notebook


In [ ]:
from course_utils.server_launcher import create_server_launcher
api_url = 'http://127.0.0.1:8098'
server_command = [sys.executable, str(education_dir/'preparation/serve_transolver.py'),
                  '--training-dir', str(training_dir.resolve()), '--surface-dir', str(surface_dir.resolve()),
                  '--device', device, '--port', '8098']
server_command_text = subprocess.list2cmdline(server_command)
server_bat = create_server_launcher(server_dir/'start_inference_server.bat', server_command, education_dir.parent)
print('서버 실행 파일:', server_bat)
print('파일이 있는 폴더:', server_bat.parent)
print('파일 탐색기에서 BAT 파일을 더블클릭한 뒤 다음 셀로 진행하세요.')


### 4.2 서버의 준비 상태 확인하기


### 07장 / 셀 17 / Notebook


In [ ]:
sys.path.insert(0, str(education_dir/'omniverse'))
from client import HookClient
client = HookClient(api_url)
health = client.request('/health')
assert health['ready'] and health.get('api_kind') == 'live_transolver'
assert health['device'] == device and health['model_id'] == context['checkpoint_sha256']
catalog = client.request('/geometries')
print('서버 준비:', health['ready'], '| 장치:', health['device'])
print('6장과 같은 모델:', health['model_id'] == context['checkpoint_sha256'])
print('연속 두께 범위(mm):', catalog['thickness_range_mm'])
print('API 문서:', api_url+'/docs')


## 5. 두께/하중을 보내 예측 결과 받기


### 5.1 첫 요청을 보내고 응답 읽기


### 07장 / 셀 20 / Notebook


In [ ]:
surface, prediction, timing = client.fetch_conditions(**request_body)
print('적용 조건:', prediction['thickness_mm'], 'mm /', prediction['force_N'], 'N')
print('전체 계산 노드:', prediction['full_node_count'], '| 응답 표면 노드:', len(prediction['node_ids']))
print('응력 배열:', np.asarray(prediction['stress_mpa']).shape,
      '| 변위 배열:', np.asarray(prediction['displacement_mm']).shape)
print('단위:', prediction['units'])
print('앞 세 노드 ID:', prediction['node_ids'][:3])
print('앞 세 응력(MPa):', np.round(prediction['stress_mpa'][:3], 4))
print('앞 세 변위(mm):', np.round(prediction['displacement_mm'][:3], 6))


### 5.2 예측값이 같은 노드에 전달됐는지 확인하기


### 07장 / 셀 23 / Notebook


In [ ]:
# 1. 첫 요청과 같은 조건으로 전체 노드를 직접 추론합니다.
case = cases[prediction['case_id']]
morphed_coords = morph_points(case.coords, morphing_manifest['reference_thickness_mm'], prediction['thickness_mm'])
direct = predict_fields(model, normalizer, morphed_coords,
                        np.array([prediction['thickness_mm'], prediction['force_N']]))
# 2. 노드 ID로 표면과 전체 결과의 위치를 맞춥니다.
lookup = {int(node): i for i, node in enumerate(case.node_ids)}
indices = [lookup[int(node)] for node in surface['node_ids']]
np.testing.assert_allclose(surface['points_mm'], morphed_coords[indices], rtol=0, atol=1e-5)
assert prediction['node_ids'] == surface['node_ids']
assert prediction['units']['stress'] == 'MPa' and prediction['units']['displacement'] == 'mm'
# 3. 같은 노드의 응력과 변위 성분 차이를 계산합니다.
stress_difference = float(np.max(np.abs(np.asarray(prediction['stress_mpa']) - direct['stress'][indices, 0])))
displacement_difference = float(np.max(np.abs(np.asarray(prediction['displacement_mm']) - direct['displacement'][indices])))
assert stress_difference <= 1e-4 and displacement_difference <= 1e-6
print('직접 추론과 응력 최대 차이(MPa):', stress_difference)
print('직접 추론과 변위 성분 최대 차이(mm):', displacement_difference)
print('모델/단위/표면 위치/노드 순서 확인 완료')


### 5.3 같은 하중에서 6 mm와 12 mm 비교하기


### 07장 / 셀 26 / Notebook


In [ ]:
fixed_force = request_body['force_N']
thin_body = {'thickness_mm': 6.0, 'force_N': fixed_force}
changed_body = {'thickness_mm': 12.0, 'force_N': fixed_force}
thin_surface, thin_prediction, _ = client.fetch_conditions(**thin_body)
changed_surface, changed_prediction, _ = client.fetch_conditions(**changed_body)
assert thin_prediction['force_N'] == changed_prediction['force_N'] == fixed_force
assert thin_prediction['node_ids'] == changed_prediction['node_ids'] == prediction['node_ids']
assert thin_prediction['model_id'] == changed_prediction['model_id'] == prediction['model_id']
assert thin_prediction['surface_version'] != changed_prediction['surface_version']
for body, geometry, result in [(thin_body, thin_surface, thin_prediction),
                               (changed_body, changed_surface, changed_prediction)]:
    actual_thickness = np.ptp(np.asarray(geometry['points_mm'])[:, 2])
    assert np.isclose(actual_thickness, body['thickness_mm'])
    assert isinstance(result['cache_hit'], bool)  # 같은 조건의 재실행도 허용
print('공통 하중(N):', fixed_force)
print('두께(mm) | 실제 형상 두께(mm) | 전체 최대응력(MPa) | 전체 최대 변위 크기(mm) | 학습 영역 구분')
for geometry, result in [(thin_surface, thin_prediction), (changed_surface, changed_prediction)]:
    print(f"{result['thickness_mm']:g} | {np.ptp(np.asarray(geometry['points_mm'])[:, 2]):g} | "
          f"{result['full_max_stress_mpa']:.6f} | {result['full_max_displacement_mm']:.6f} | "
          f"{'학습 영역 안' if result['inside_training_hull'] else '외삽(학습 영역 밖)'}")
print('최대응력 차이: 6 mm − 12 mm (MPa):',
      thin_prediction['full_max_stress_mpa'] - changed_prediction['full_max_stress_mpa'])
print('최대 변위 크기 차이: 6 mm − 12 mm (mm):',
      thin_prediction['full_max_displacement_mm'] - changed_prediction['full_max_displacement_mm'])
from course_utils.morphing_plot import plot_thickness_predictions
from IPython.display import Image, display
thickness_figure = plot_thickness_predictions(
    [thin_surface, changed_surface], [thin_prediction, changed_prediction],
    server_dir/'thickness_prediction_comparison.png')
display(Image(filename=thickness_figure['image']))


### 5.4 설계 범위를 벗어난 입력 확인하기


### 07장 / 셀 29 / Notebook


In [ ]:
import urllib.error
try:
    client.request('/predict', {'thickness_mm': 999, 'force_N': 300})
    raise AssertionError('범위 밖의 두께가 허용되었습니다.')
except urllib.error.HTTPError as error:
    assert error.code == 422
    error_detail = json.loads(error.read().decode('utf-8'))['detail']
    print('의도한 입력 오류 응답:', error.code)
    print('입력 오류 안내:', error_detail[0]['msg'])
    print('두께 6~12 mm, 하중 100~500 N 범위로 다시 요청하세요.')


## 6. Omniverse 연결에 필요한 정보 정리하기


### 6.1 서버 주소와 실행 정보 저장하기


### 07장 / 셀 32 / Notebook


In [ ]:
connection = dict(api_url=api_url, training_dir=str(training_dir.resolve()), surface_dir=str(surface_dir.resolve()),
                  model_sha256=context['checkpoint_sha256'], server_command=server_command_text,
                  server_launcher=str(server_bat.resolve()),
                  request_example=request_body)
api_check = dict(passed=True, context=context, api_url=api_url, device=health['device'], real_http=True,
                 request=request_body, thin_request=thin_body, changed_request=changed_body,
                 stress_max_difference_mpa=stress_difference, displacement_max_difference_mm=displacement_difference,
                 node_order_checked=True, units_checked=True, out_of_range_thickness_status=422,
                 purpose='Inference delivery check; not ANSYS accuracy evaluation')
for name, value in [('inference_connection.json', connection), ('inference_api_check.json', api_check)]:
    (server_dir/name).write_text(json.dumps(value, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
print('연결 정보:', server_dir/'inference_connection.json')
print('전달 확인 기록:', server_dir/'inference_api_check.json')
from course_utils.server_handoff import save_server_handoff
chapter7_handoff = save_server_handoff(education_dir, server_dir)
print('8장 연결 준비 완료:', chapter7_handoff)
print('8장 API 주소:', api_url)
print('서버 Console은 8장에서도 열어 두세요.')
